In [ ]:
!pip install tensorflow
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 80.4 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import time
# Mengganti pickle dengan joblib
import joblib
import numpy as np
import pandas as pd
import tensorflow as tf
from transformers import TFBertModel, BertTokenizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import faiss
from tqdm import tqdm
from scipy.stats import mode

# KONFIGURASI
MODEL_NAME = "bert-base-multilingual-cased"
MAX_LEN = 128
BATCH_SIZE = 64
K_NEIGHBORS = 10
ARTIFACTS_DIR = "/content/drive/My Drive/colab/bert_faiss_artifacts"

# 1.LOAD & PREPARE DATA 
def prepare_data(file_path):
    print(f"Loading data = {file_path}...")
    df = pd.read_csv(file_path)
    df = df.dropna(subset=['text', 'label', 'bahasa'])

    # Tambah prefix bahasa
    df['text'] = df['bahasa'].apply(lambda x: '[ID]' if x.lower() == 'in' else '[EN]') + ' ' + df['text']

    texts = df['text'].tolist()
    labels = df['label'].astype(int).tolist()

    # Split data menjadi train dan test
    X_train, X_val, y_train, y_val = train_test_split(
        texts, labels,
        test_size=0.2,
        stratify=labels,
        random_state=42
    )
    print(f"Train size: {len(X_train)}, Validation size: {len(X_val)}")
    return X_train, X_val, np.array(y_train), np.array(y_val)

# 2.HASILKAN EMBEDDING 
def get_bert_embeddings(texts, model, tokenizer):
    print(f"Generate BERT embeddings for {len(texts)} texts...")
    all_embeddings = []

    # Proses dalam batch untuk efisiensi memori
    for i in tqdm(range(0, len(texts), BATCH_SIZE)):
        batch_texts = texts[i:i + BATCH_SIZE]
        inputs = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=MAX_LEN,
            return_tensors="tf"
        )

        outputs = model(inputs)

        # ambil [CLS] token embedding (last_hidden_state[:, 0, :])
        cls_embeddings = outputs.last_hidden_state[:, 0, :].numpy()
        all_embeddings.append(cls_embeddings)

    return np.vstack(all_embeddings)

# 3.SAVE ARTIFACTS
def save_artifacts(train_embeddings, val_embeddings, train_labels, faiss_index):
    print(f"Saving artifacts to {ARTIFACTS_DIR}...")
    os.makedirs(ARTIFACTS_DIR, exist_ok=True)

    #simpan embedding
    np.save(os.path.join(ARTIFACTS_DIR, "train_embeddings.npy"), train_embeddings)
    np.save(os.path.join(ARTIFACTS_DIR, "validation_embeddings.npy"), val_embeddings)
    print("Embedding saved sucess.")

    #simpan label
    joblib.dump(train_labels, os.path.join(ARTIFACTS_DIR, "train_labels.joblib"))
    print("Train labels saved sucess.")

    #simpan indeks FAISS
    faiss.write_index(faiss_index, os.path.join(ARTIFACTS_DIR, "faiss_index.index"))
    print("FAISS index saved success.")


# 4.KNN(FAISS) PROCESS
def predict_knn_faiss(val_embeddings, faiss_index, train_labels):
    print("Starting KNN(FAISS) clasification...")

    start_time = time.time()
    _, neighbor_indices = faiss_index.search(val_embeddings, K_NEIGHBORS)
    neighbor_labels = train_labels[neighbor_indices]
    predictions, _ = mode(neighbor_labels, axis=1)
    end_time = time.time()
    inference_time = end_time - start_time

    print(f"Proses waktu yang dibutuhkan {inference_time:.4f} seconds.")
    return predictions.flatten(), inference_time

# RUN
def main(data_path):
    X_train, X_val, y_train, y_val = prepare_data(data_path)

    # Muat model BERT dasar dan tokenizer
    print(f"Loading BERT model ({MODEL_NAME})...")
    tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)
    bert_model = TFBertModel.from_pretrained(MODEL_NAME, from_pt=True)

    # Hasilkan Embedding
    train_embeddings = get_bert_embeddings(X_train, bert_model, tokenizer)
    val_embeddings = get_bert_embeddings(X_val, bert_model, tokenizer)

    # Pastikan tipe data float32
    train_embeddings = train_embeddings.astype('float32')
    val_embeddings = val_embeddings.astype('float32')

    print(f"Shape of train embeddings: {train_embeddings.shape}")
    print(f"Shape of validation embeddings: {val_embeddings.shape}")

    # Indexing dengan FAISS
    print("Creating and building FAISS index...")
    embedding_dim = train_embeddings.shape[1]
    index = faiss.IndexFlatL2(embedding_dim) #L2 distance
    index.add(train_embeddings)
    print(f"FAISS index created. Total vectors in index: {index.ntotal}")

    # Simpan semua artefak
    save_artifacts(train_embeddings, val_embeddings, y_train, index)

    # Klasifikasi dan evaluasi
    predictions, inference_time = predict_knn_faiss(val_embeddings, index, y_train)

    print("\n" + "="*20 + " HASIL EVALUASI " + "="*20)
    print(classification_report(y_val, predictions, target_names=['Label 0', 'Label 1']))
    print(f"Total Inference Time: {inference_time:.4f} seconds")
    avg_inference_time = (inference_time / len(y_val)) * 1000
    print(f"Rata-rata inference waktu per sampel: {avg_inference_time:.4f} ms")
    print("=" * 60)

# EKSEKUSI
if __name__ == "__main__":
    DATASET_PATH = "/content/drive/My Drive/dataset/4datasentimen_all_clean_ing_nolowercase_googlecolaboutlier34.csv"
    main(DATASET_PATH)


[INFO] Loading and preparing data from /content/drive/My Drive/dataset/4datasentimen_all_clean_ing_nolowercase_googlecolaboutlier34.csv...
[INFO] Data prepared. Train size: 5524, Validation size: 1381
[INFO] Loading BERT model (bert-base-multilingual-cased)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

...

[INFO] Generating BERT embeddings for 5524 texts...


100%|██████████| 87/87 [00:20<00:00,  4.27it/s]


[INFO] Generating BERT embeddings for 1381 texts...


100%|██████████| 22/22 [00:05<00:00,  4.28it/s]


[INFO] Shape of train embeddings: (5524, 768)
[INFO] Shape of validation embeddings: (1381, 768)
[INFO] Creating and building FAISS index...
[INFO] FAISS index created. Total vectors in index: 5524
[INFO] Saving artifacts to /content/drive/My Drive/colab/bert_faiss_artifacts...
[INFO] Embeddings saved.
[INFO] Train labels saved with joblib.
[INFO] FAISS index saved.
--------------------------------------------------
[INFO] Starting KNN classification using FAISS index...
[INFO] Inference finished in 0.1312 seconds.

==================== EVALUATION REPORT ====================
              precision    recall  f1-score   support

     Label 0       0.88      0.84      0.86       860
     Label 1       0.75      0.81      0.78       521

    accuracy                           0.83      1381
   macro avg       0.82      0.82      0.82      1381
weighted avg       0.83      0.83      0.83      1381

Total Inference Time: 0.1312 seconds
Average Inference Time per Sample: 0.0950 ms


In [ ]:
import os
import time
import joblib
import numpy as np
import pandas as pd
import tensorflow as tf
from transformers import TFBertModel, BertTokenizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.linear_model import LogisticRegression
from tqdm import tqdm

# ======================== KONFIGURASI ========================
# PASTIKAN PATH INI SESUAI DENGAN LOKASI ARTEFAK ANDA DI GOOGLE DRIVE
ARTIFACTS_DIR = "/content/drive/My Drive/colab/bert_faiss_knn_artifacts"
MODEL_NAME = "bert-base-multilingual-cased"
MAX_LEN = 128
K_NEIGHBORS = 10

# ======================== FUNGSI PREDIKSI ========================
def predict_single_text(text_to_predict, tokenizer, bert_model, faiss_index, train_labels):
    """
    Fungsi untuk mengklasifikasikan satu buah teks.
    """
    print(f"Analyzing text: '{text_to_predict}'...")
    start_time = time.time()

    # 1. Buat embedding dari teks input
    inputs = tokenizer(
        [text_to_predict],
        padding=True,
        truncation=True,
        max_length=MAX_LEN,
        return_tensors="tf"
    )
    outputs = bert_model(inputs)
    embedding = outputs.last_hidden_state[:, 0, :].numpy()
    embedding = embedding.astype('float32')

    # 2. Cari tetangga terdekat di indeks FAISS
    distances, indices = faiss_index.search(embedding, K_NEIGHBORS)

    # 3. Ambil label dari tetangga dan lakukan majority voting
    neighbor_indices = indices[0]
    neighbor_labels = train_labels[neighbor_indices]

    prediction_array, count_array = mode(neighbor_labels, keepdims=False)
    predicted_label = int(prediction_array)
    vote_count = int(count_array)

    # prediction_array, count_array = mode(neighbor_labels)
    # predicted_label = int(prediction_array[0])
    # vote_count = int(count_array[0])

    confidence = vote_count / K_NEIGHBORS
    end_time = time.time()

    # 4. Tampilkan hasil analisis
    print("\n" + "="*20 + " ANALYSIS RESULT " + "="*20)
    print(f"Predicted Class: Label = {predicted_label}")
    print(f"Confidence: {confidence:.2%} ({vote_count} out of {K_NEIGHBORS} neighbors voted for this class)")
    print(f"Labels of Nearest Neighbors Found: {neighbor_labels}")
    print(f"Inference Time: {end_time - start_time:.4f} seconds")
    print("=" * 57)

    return predicted_label

# ======================== SCRIPT UTAMA ========================
if __name__ == "__main__":
    print("[INFO] Loading necessary artifacts, please wait...")

    try:
        # Muat semua artefak yang diperlukan
        tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)
        bert_model = TFBertModel.from_pretrained(MODEL_NAME, from_pt=True)

        faiss_index_path = os.path.join(ARTIFACTS_DIR, "faiss_index.index")
        faiss_index = faiss.read_index(faiss_index_path)

        # --- PERUBAHAN DI SINI ---
        # Menggunakan joblib untuk memuat file train_labels.joblib
        train_labels_path = os.path.join(ARTIFACTS_DIR, "train_labels.joblib")
        train_labels = np.array(joblib.load(train_labels_path))
        # -------------------------

        print("[SUCCESS] All artifacts loaded. Ready to predict.")
        print("-" * 57)

        # Teks yang ingin dideteksi
        input_text = "stupid system"

        # Panggil fungsi prediksi
        predict_single_text(input_text, tokenizer, bert_model, faiss_index, train_labels)

    except FileNotFoundError as e:
        print(f"\n[ERROR] Artifact file not found: {e}")
        print("Please make sure the path in ARTIFACTS_DIR is correct and your Google Drive is mounted.")
    except Exception as e:
        print(f"\n[ERROR] An unexpected error occurred: {e}")

[INFO] Loading necessary artifacts, please wait...


Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['cls.predictions.bias', 'cls.seq_relationship.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.dense.bias', 'cls.predictions.decoder.weight', 'cls.seq_relationship.weight', 'cls.predictions.transform.dense.weight']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFBertModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already

[SUCCESS] All artifacts loaded. Ready to predict.
---------------------------------------------------------
Analyzing text: 'stupid system'...

==================== ANALYSIS RESULT ====================
Predicted Class: Label 1
Confidence: 100.00% (10 out of 10 neighbors voted for this class)
Labels of Nearest Neighbors Found: [1 1 1 1 1 1 1 1 1 1]
Inference Time: 0.2068 seconds
